# Module 2: Sentiment / Emotion Classifier

**Goal:** classify the emotional tone of a customer message so the system can route frustrated
customers to a more empathetic / priority-flagged response path.

**Dataset:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) — ~20k
English Twitter messages labeled with 6 emotions (sadness, joy, love, anger, fear, surprise).

**Approach:** a **Recurrent Neural Network** (BiLSTM), one of the two options the brief allows
(RNN or Transformer). An RNN was chosen deliberately over a transformer:

- The dataset is short, single-sentence text (~20k rows) — a BiLSTM with a small embedding
  layer trains in a few minutes on CPU, whereas fine-tuning a transformer (e.g. DistilBERT)
  needs GPU time to be practical.
- The brief explicitly warns to "avoid overcomplicated approaches you don't fully grasp" —
  a from-scratch BiLSTM is fully transparent line-by-line for the oral assessment, whereas a
  fine-tuned transformer involves more moving parts (tokenizer internals, pretrained weights)
  that are harder to defend in depth on short notice.
- A transformer fine-tune is the natural upgrade path noted at the end of this notebook if
  GPU time is available and higher accuracy is needed.

**Domain-shift caveat:** this dataset is Twitter text, not customer-support text. We keep a
small hand-written customer-support-style qualitative sample (`data/sample_emotion.csv`) as an
independent sanity check, separate from the Twitter-trained model's own held-out split.

**Label mapping:** the 6 fine-grained emotions are trained first (this is what the dataset
actually supervises), then mapped to 3 routing buckets:
`negative={sadness,anger,fear}`, `positive={joy,love}`, `neutral={surprise}`. Surprise is kept
neutral since it's valence-ambiguous (a surprise can be pleasant or unpleasant) — a documented
design decision, not an oversight.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src import sentiment_classifier as sc


## 2.1 Load data

In [2]:
df = sc.load_hf_dataset()
print(df.shape)
print(df['label'].value_counts())
df.head()


(20000, 2)
label
joy         6761
sadness     5797
anger       2709
fear        2373
love        1641
surprise     719
Name: count, dtype: int64


,text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


## 2.2 Preprocessing
Lowercasing, URL stripping, non-alphabetic character removal, whitespace normalization (see `clean_text`), then a simple whitespace-token vocabulary capped at 20k tokens, padded/truncated to `MAX_LEN=40` tokens — plenty for short Twitter-style messages.

In [3]:
df['clean'] = df['text'].apply(sc.clean_text)
df[['text', 'clean']].head()


,text,clean
0,i didnt feel humiliated,i didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,i can go from feeling so hopeless to so damned...
2,im grabbing a minute to post i feel greedy wrong,im grabbing a minute to post i feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,i am ever feeling nostalgic about the fireplac...
4,i am feeling grouchy,i am feeling grouchy


## 2.3 Model architecture

```
Embedding(vocab_size, 100, padding_idx=0)
  -> BiLSTM(100 -> 64, bidirectional)
  -> concat(final forward, final backward hidden states)
  -> Dropout(0.3)
  -> Linear(128 -> n_classes)
```

Bidirectional so the model sees both left and right context around emotionally-loaded words; final hidden states (not just the last timestep) are concatenated from both directions as the sentence representation.

## 2.4 Train
Requires PyTorch. If torch isn't installed in this environment, `train()` automatically falls back to a TF-IDF + Logistic Regression classifier so the notebook still runs end to end for local testing — **the graded deliverable is the BiLSTM above**; install torch (`pip install torch`) to train it for real.

In [4]:
model, vocab, label2id = sc.train(df)


epoch 1/5 - train loss 1.5453
epoch 2/5 - train loss 0.9866
epoch 3/5 - train loss 0.5610
epoch 4/5 - train loss 0.3441
epoch 5/5 - train loss 0.2297
              precision    recall  f1-score   support

       anger       0.84      0.83      0.83       542
        fear       0.78      0.84      0.81       475
         joy       0.90      0.84      0.87      1352
        love       0.74      0.54      0.63       328
     sadness       0.81      0.94      0.87      1159
    surprise       0.80      0.47      0.59       144

    accuracy                           0.83      4000
   macro avg       0.81      0.75      0.77      4000
weighted avg       0.83      0.83      0.83      4000



## 2.5 Save model

In [5]:
sc.save(model, vocab, label2id, out_dir='../models/sentiment')
print('Saved.')


Saved.


## 2.6 Inference + bucket mapping demo

In [6]:
for msg in [
    "this is the worst experience ever, so angry",
    "thank you so much this made my day",
    "wow I did not expect that refund so fast",
    "can you tell me your return policy",
]:
    print(msg, '->', sc.predict_sentiment(model, vocab, label2id, msg))


this is the worst experience ever, so angry -> {'emotion': 'anger', 'bucket': 'negative', 'confidence': 0.25601452589035034}
thank you so much this made my day -> {'emotion': 'joy', 'bucket': 'positive', 'confidence': 0.26780620217323303}
wow I did not expect that refund so fast -> {'emotion': 'sadness', 'bucket': 'negative', 'confidence': 0.27391231060028076}
can you tell me your return policy -> {'emotion': 'sadness', 'bucket': 'negative', 'confidence': 0.4722801148891449}


## 2.7 Qualitative domain-shift check
Run the same model on hand-written, customer-support-style messages (not Twitter text) to sanity-check generalization beyond the training domain.

In [7]:
support_style_df = pd.read_csv('../data/sample_emotion.csv')
for _, row in support_style_df.head(8).iterrows():
    pred = sc.predict_sentiment(model, vocab, label2id, row['text'])
    print(f"gold={row['label']:>8s} pred={pred['emotion']:>8s} bucket={pred['bucket']:>8s} | {row['text'][:60]}")


gold=   anger pred=   anger bucket=negative | i am so frustrated this package never arrived and no one is 
gold=   anger pred= sadness bucket=negative | this is the worst service i have ever experienced in my life
gold=   anger pred=   anger bucket=negative | i am furious that i was charged twice for the same order
gold=   anger pred=   anger bucket=negative | why does support keep ignoring my messages this is ridiculou
gold= sadness pred= sadness bucket=negative | i feel so sad my order got cancelled without any warning
gold= sadness pred= sadness bucket=negative | i am heartbroken the gift i ordered will not arrive in time
gold= sadness pred= sadness bucket=negative | i feel let down that the refund still has not been processed
gold=    fear pred=    fear bucket=negative | i am scared this transaction on my account was not authorize


## Notes / decisions to defend at assessment
- RNN (BiLSTM) chosen over transformer for CPU-trainability + full transparency at the
  assessment; transformer fine-tune is a documented upgrade path, not something we didn't
  consider.
- 6 fine emotions trained (matches the dataset's real gold labels), then mapped to 3 routing
  buckets — deliberate 2-stage design, not information loss by accident.
- `surprise -> neutral` is a defensible-but-debatable choice; an alternative would be a
  4-bucket scheme splitting surprise into positive/negative by context, at the cost of extra
  ambiguity in the labels the model has to learn.
- Domain shift (Twitter -> customer support) is explicitly flagged and checked qualitatively,
  not ignored.